# Notebook 10 – Imbalanced Classification

## What is Class Imbalance?
Class imbalance occurs when one class (the majority class) significantly outweighs the other class (the minority class) in a dataset. 
- **Example:** Fraud detection (99% normal transactions, 1% fraud).

### Why Accuracy is Misleading
In an imbalanced dataset, accuracy can be deceptive. If 99% of your data is "Normal", a model that predicts "Normal" for every single case will have **99% accuracy**, but it completely fails to detect the fraud (0% recall for the minority class).

### Better Metrics for Imbalanced Data:
- **Precision:** Out of all predicted positives, how many were actually positive?
- **Recall (Sensitivity):** Out of all actual positives, how many did we catch?
- **F1-Score:** The harmonic mean of Precision and Recall.
- **ROC-AUC:** Area under the Receiver Operating Characteristic curve. It measures the model's ability to distinguish between classes.
- **PR-AUC:** Area under the Precision-Recall curve. More reliable than ROC-AUC for highly imbalanced data.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, recall_score

# Create an imbalanced dataset (90% Class 0, 10% Class 1)
X, y = make_classification(n_samples=1000, n_features=20, weights=[0.9, 0.1], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Baseline model (no handling of imbalance)
model_baseline = RandomForestClassifier(random_state=42)
model_baseline.fit(X_train, y_train)
y_pred = model_baseline.predict(X_test)

print("--- Baseline Model (Imbalanced) ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

--- Baseline Model (Imbalanced) ---
Accuracy: 0.9400

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.99      0.97       180
           1       0.90      0.45      0.60        20

    accuracy                           0.94       200
   macro avg       0.92      0.72      0.78       200
weighted avg       0.94      0.94      0.93       200



## Approaches to Handling Imbalance

### 1. Class Weights
Most scikit-learn models have a `class_weight` parameter. Setting it to `'balanced'` tells the model to penalize mistakes on the minority class more heavily.

### 2. Undersampling
Removing samples from the majority class to balance the ratio.
- **Pro:** Reduces training time.
- **Con:** Throws away potentially useful information.

### 3. Oversampling & SMOTE
- **Random Oversampling:** Duplicating samples from the minority class.
- **SMOTE (Synthetic Minority Over-sampling Technique):** Instead of duplicating, it creates *synthetic* new samples by interpolating between existing minority samples.

In [4]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# 1. Using Class Weights
model_weighted = RandomForestClassifier(class_weight='balanced', random_state=42)
model_weighted.fit(X_train, y_train)
y_pred_weighted = model_weighted.predict(X_test)

# 2. Using SMOTE (Oversampling)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)
model_smote = RandomForestClassifier(random_state=42)
model_smote.fit(X_resampled, y_resampled)
y_pred_smote = model_smote.predict(X_test)

# 3. Using Undersampling
rus = RandomUnderSampler(random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train)
model_under = RandomForestClassifier(random_state=42)
model_under.fit(X_under, y_under)
y_pred_under = model_under.predict(X_test)

print("--- Comparison ---")
print(f"Baseline Recall: {recall_score(y_test, y_pred):.4f}")
print(f"Class Weights Recall: {recall_score(y_test, y_pred_weighted):.4f}")
print(f"SMOTE Recall: {recall_score(y_test, y_pred_smote):.4f}")
print(f"Undersampling Recall: {recall_score(y_test, y_pred_under):.4f}")

--- Comparison ---
Baseline Recall: 0.4500
Class Weights Recall: 0.6500
SMOTE Recall: 0.6500
Undersampling Recall: 0.8000


## Summary of Approaches
- **Class Weights:** Easiest and fastest, doesn't change the data.
- **SMOTE:** Usually the best for improving recall, but can introduce noise.
- **Undersampling:** Best for massive datasets where you need to reduce size.
- **Threshold Adjustment:** Combine these methods with threshold tuning (from Notebook 9) for maximum control.